In [ ]:
!pip install -q sentence-transformers scikit-learn pandas

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
sentences = [
    "Aspirin reduces the risk of heart attack.",
    "Taking aspirin can lower the chance of myocardial infarction.",
    "The Eiffel Tower is located in Paris.",
    "Paris is home to the Eiffel Tower.",
    "Neural networks are used for machine learning.",
    "Deep learning models rely on neural networks.",
    "Bananas are yellow fruits.",
    "A car engine requires fuel.",
]

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
embeddings = model.encode(sentences, normalize_embeddings=True)

In [ ]:
query = "Aspirin helps prevent heart attacks."
query_embedding = model.encode([query], normalize_embeddings=True)

In [ ]:
scores = cosine_similarity(query_embedding, embeddings)[0]

In [ ]:
results = pd.DataFrame({
    "sentence": sentences,
    "cosine_similarity": scores
}).sort_values("cosine_similarity", ascending=False)

In [ ]:
results

In [ ]:
!pip install -q beir

In [ ]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

In [ ]:
dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

In [ ]:
data_path = util.download_and_unzip(url, "datasets")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")

In [ ]:
print(type(corpus))
print(type(queries))
print(type(qrels))

In [ ]:
print("Corpus size:", len(corpus))
print("Queries size:", len(queries))
print("Qrels size:", len(qrels))

corpus_df = pd.DataFrame.from_dict(corpus, orient="index").reset_index()
corpus_df = corpus_df.rename(columns={"index": "_id"})

queries_df = pd.DataFrame.from_dict(queries, orient="index", columns=["text"]).reset_index()
queries_df = queries_df.rename(columns={"index": "_id"})

qrels_rows = []

for query_id, relevant_docs in qrels.items():
    for doc_id, score in relevant_docs.items():
        qrels_rows.append({"query-id": query_id, "corpus-id": doc_id, "score": score})

qrels_df = pd.DataFrame(qrels_rows)
display(corpus_df.head())
display(queries_df.head())
display(qrels_df.head())

In [ ]:
query_overlap = qrels_df["query-id"].isin(queries_df["_id"]).mean()
corpus_overlap = qrels_df["corpus-id"].isin(corpus_df["_id"]).mean()

print("Query overlap:", query_overlap)
print("Corpus overlap:", corpus_overlap)

In [ ]:
for i in range(5):
    row = qrels_df.iloc[i]

    query_id = row["query-id"]
    corpus_id = row["corpus-id"]

    query_text = queries_df.loc[queries_df["_id"] == query_id, "text"].iloc[0]
    doc = corpus_df.loc[corpus_df["_id"] == corpus_id].iloc[0]

    print("=" * 100)
    print("QUERY ID:", query_id)
    print("QUERY:", query_text)
    print()
    print("RELEVANT DOC ID:", corpus_id)
    print("TITLE:", doc["title"])
    print("TEXT:", doc["text"][:1000])
    print()

In [ ]:
!pip install -q chromadb sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
documents = []
metadatas = []
ids = []

for _, row in corpus_df.iterrows():
    doc_text = f"{row['title']} {row['text']}"
    documents.append(doc_text)
    metadatas.append({"title": row["title"]})
    ids.append(str(row["_id"]))

In [ ]:
subset_size = 500

subset_docs = documents[:subset_size]
subset_ids = ids[:subset_size]
subset_metadata = metadatas[:subset_size]

In [ ]:
doc_embeddings = embedding_model.encode(
    subset_docs,
    show_progress_bar=True,
    normalize_embeddings=True
)

In [ ]:
import chromadb

client = chromadb.Client()

collection = client.create_collection(name="scifact")

In [ ]:
collection.add(
    documents=subset_docs,
    embeddings=doc_embeddings.tolist(),
    metadatas=subset_metadata,
    ids=subset_ids
)

In [ ]:
def search(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results

In [ ]:
query = "Does aspirin reduce heart attack risk?"

results = search(query)

for i in range(len(results["documents"][0])):
    print("=" * 80)
    print("DOC ID:", results["ids"][0][i])
    print("TITLE:", results["metadatas"][0][i]["title"])
    print()
    print(results["documents"][0][i][:1000])

In [ ]:
queries_to_test = [
    "COVID vaccine effectiveness",
    "brain cancer treatment",
    "heart disease prevention",
    "gene mutation effects",
    "protein folding"
]

for q in queries_to_test:
    print("
" + "=" * 100)
    print("QUERY:", q)

    results = search(q, top_k=3)

    for i in range(3):
        print("
--- RESULT", i + 1)
        print("TITLE:", results["metadatas"][0][i]["title"])
        print(results["documents"][0][i][:500])